# Power analysis on real production data

This notebook walks through the canonical `absim` workflow for someone who already has historical metric values from their warehouse and wants to know:

1. **Are the candidate criteria calibrated on *my* distribution?** (FPR ≈ α under H₀.)
2. **Which criterion gives me the most power for the lift I expect to ship?**

We use `EmpiricalGenerator` to bootstrap-resample a real per-user revenue array and inject calibrated effects into the treatment arm — so the simulation reflects the actual zero-inflation and heavy tails of a production payments metric, not a textbook Gaussian.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from absim import EffectSize, Simulator
from absim.criteria import Bootstrap, WelchTTest
from absim.generators import EmpiricalGenerator

sns.set_theme(style="whitegrid", context="notebook")
RNG = np.random.default_rng(seed=20260506)

## 1. Stand in for a warehouse pull

In production this cell is replaced by a `pd.read_parquet("revenue_last_month.parquet")`. For reproducibility we synthesize a representative one-month sample with the structural quirks every payments metric has: a heavy point mass at zero (non-payers) plus a long lognormal right tail.

In [ ]:
n_users = 50_000
share_payers = 0.18
is_payer = RNG.uniform(size=n_users) < share_payers
revenue = np.where(is_payer, RNG.lognormal(mean=2.6, sigma=1.0, size=n_users), 0.0)

summary = pd.Series(revenue).describe(percentiles=[0.5, 0.9, 0.95, 0.99]).round(2)
print(summary.to_string())

fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(revenue[revenue > 0], bins=60, color="#3b6e8f")
ax.set_yscale("log")
ax.set_xlabel("Per-user revenue (paying users only)")
ax.set_ylabel("Count (log)")
ax.set_title("Synthetic 'warehouse' sample — zero-inflated lognormal")
fig.tight_layout()

## 2. Calibration audit (H₀)

Before trusting any criterion's power, verify that on **this** distribution it actually rejects α = 5% of the time under H₀. We simulate 3 000 A/A iterations and read the Wilson 95% CI on the rejection rate; if the CI brackets 0.05, the criterion is calibrated.

In [ ]:
gen = EmpiricalGenerator(outcomes=revenue, n_per_group=5_000, relative=True)
criteria = [
    WelchTTest(),
    Bootstrap(method="percentile", n_resamples=1_000),
    Bootstrap(method="bca", n_resamples=1_000),
]

rows = []
for crit in criteria:
    label = f"{crit.name}/{getattr(crit, 'method', '')}".rstrip("/")
    sim = Simulator(gen, crit, n_sims=3_000, effect=EffectSize("none", 0.0), seed=0)
    r = sim.run()
    rows.append({
        "criterion": label,
        "FPR": round(r.fpr, 4),
        "CI_low": round(r.binomial_ci_low, 4),
        "CI_high": round(r.binomial_ci_high, 4),
        "calibrated?": "yes" if r.binomial_ci_low <= 0.05 <= r.binomial_ci_high else "NO",
    })
pd.DataFrame(rows)

Read this table column-by-column. If `CI_low ≤ 0.05 ≤ CI_high`, the criterion is calibrated on *this* metric. Heavy-tailed Welch sometimes drifts to FPR ≈ 0.06–0.07 — that's the kind of finding that would change your team's choice of test.

## 3. Power sweep — what lift can we detect?

We sweep relative lifts from 0% to 8% and ask each criterion how often it correctly rejects H₀.

In [ ]:
lifts = [0.00, 0.01, 0.02, 0.03, 0.05, 0.08]
criteria_for_power = [
    (WelchTTest(), "welch_t"),
    (Bootstrap(method="bca", n_resamples=1_000), "bootstrap_bca"),
]

records = []
for crit, label in criteria_for_power:
    for lift in lifts:
        sim = Simulator(gen, crit, n_sims=2_000,
                        effect=EffectSize(f"+{lift:.0%}", lift), seed=0)
        r = sim.run()
        records.append({
            "criterion": label,
            "lift": lift,
            "power": r.rejection_rate,
            "ci_low": r.binomial_ci_low,
            "ci_high": r.binomial_ci_high,
        })
df = pd.DataFrame(records)
df

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for label, sub in df.groupby("criterion"):
    ax.plot(sub["lift"], sub["power"], marker="o", label=label)
    ax.fill_between(sub["lift"], sub["ci_low"], sub["ci_high"], alpha=0.15)
ax.axhline(0.05, color="grey", lw=0.8, ls=":", label="α = 0.05")
ax.set_xlabel("Relative lift on revenue")
ax.set_ylabel("Power (rejection rate at α = 0.05)")
ax.set_title("Power on real-shaped revenue, n_per_group = 5 000")
ax.set_ylim(-0.02, 1.02)
ax.legend()
fig.tight_layout()

## 4. Reading the result

- The **flat segment near lift = 0** is each criterion's FPR — a calibration check.
- The **slope** of the curve is sensitivity. The criterion whose curve climbs faster is the one to ship.
- The **shaded band** is the Wilson 95% CI on the simulated rejection rate; bands that overlap mean the difference between criteria isn't statistically significant at the simulator's `n_sims`.

If you want a one-line summary the team can copy into the methodology RFC:

In [ ]:
target_lift = 0.03
out = (
    df.query("lift == @target_lift")
    .assign(power=lambda d: d["power"].round(3))
    .loc[:, ["criterion", "power", "ci_low", "ci_high"]]
    .reset_index(drop=True)
)
print(f"At a {target_lift:.0%} relative lift, n_per_group = 5 000:")
print(out.to_string(index=False))

## When to use which mode

- **Plenty of historical data (≥ 10 000 units), distribution looks weird** — `EmpiricalGenerator` is the right call.
- **Sparse history, but you trust a parametric shape** — `ContinuousGenerator(distribution="lognormal")` or one of the other parametric generators, with parameters fitted from the small sample.
- **Closed-form sample-size formula** — use `statsmodels.stats.power.solve_power` instead. `absim` is for the cases where the formula's assumptions don't hold.